In [ ]:
def generate_data():
	# ========== Generate simulated data ==========
	country = 'United_States'
	patterns = load_template_patterns(country, smooth=True, max_age=80)
	age_dist = load_age_distribution(country, max_age=80)
	
	# Generate the participants
	print(" Generating participants...")
	part_gen = ParticipantGenerator(age_dist['P'].values)
 
	# Calculate the number of participants per group
	n_part_grp = 250
	df_part = part_gen.generate([n_part_grp] * 2)
	print(f" Generated {len(df_part)} participants with 250 groups. \n")
	
	# Generate contact intensity matrices
	print(" Generating contact intensity matrices...")
	cint_mat_gen = ContactMatrixGenerator(patterns, age_dist['P'].values)
	cint_mat_list = [cint_mat_gen.generate(max_margin=20, seed=i) for i in range(2)]
 
	print(f" Generating contact records...")
	cnt_gen = ContactGenerator(df_part, cint_mat_list)
	df_cnt = cnt_gen.generate(seed=0)

	print(f" Generated {len(df_cnt)} contact records. \n")

	# Save the data
	print(" Saving simulation data...")
 
	# Prepare package
	data = {
		'participant': df_part,
		'contact': df_cnt,
		'age_dist': age_dist['P'].values,
		'age_dist_props': {
			'subgroup': np.ones((2, age_dist['P'].values.shape[0])) / 2
		},
		'cint_matrices': {
			'subgroup': {i: cint_mat_list[i] for i in range(len(cint_mat_list))}
		}
	}
 
	sim_data_path = 'HiBRC/fine_data.pkl'

	with open(sim_data_path, 'wb') as f:
		pickle.dump(data, f)
	
	print(f" Simulation data saved to {sim_data_path}\n")
 
	print(" Visualising simulation data...")
	# Plot the contact intensity matrices
	charts = [plot_mosaic(matrix, title=f'True Contact Intensity: Subgroup {i}') for i, matrix in enumerate(cint_mat_list)]
	chart = alt.hconcat(*charts)
	
	fig_path = 'HiBRC/fig/cint.png'
	chart.save(fig_path)
 
	# Plot marginal contact intensity
	fig_path = 'HiBRC/fig/cint_marginal.png'
	charts = [plot_mosaic_marginal(matrix.sum(axis=1), title=f'True Marginal Contact Intensity: Subgroup {i}') for i, matrix in enumerate(cint_mat_list)]
	chart = alt.hconcat(*charts)
	chart.save(fig_path)
 
	print(f" Figures saved to {fig_path}\n")

In [ ]:
import pickle

import math
import numpy as np
import pandas as pd
import os
import jax
from numpyro.infer.autoguide import AutoNormal
from numpyro.infer.initialization import init_to_value

# Get path to the root of this repository
from cntmosaic.datasets import load_template_patterns, load_age_distribution
from cntmosaic.sim import ParticipantGenerator, ContactMatrixGenerator, ContactGenerator
from cntmosaic.preprocess import make_train_data, make_full_grid
from cntmosaic.models import HiBRCfine
from cntmosaic.models.priors import PenalisedTensorSpline2D, HSGP2D
from cntmosaic.analysis import ModelSummariserSVI, ModelEvaluator, ModelVisualiser
from cntmosaic.vis import plot_mosaic, plot_mosaic_marginal, plot_mosaic_empirical
from cntmosaic.dataloader._dataloader import BaseLoader, CoordToColumns
import altair as alt

In [ ]:
with open('HiBRC/fine_data.pkl', 'rb') as f:
	data = pickle.load(f)

In [ ]:

# ===================================

# ========== Fit the model ==========
# Unpack the data    
df_part = data['participant']
df_cnt = data['contact']
age_dist = data['age_dist']
age_dist_props = data['age_dist_props']
cint_matrices = data['cint_matrices']

# Prepare training data
print(" Prepare training data...")
df_merged = pd.merge(df_cnt, df_part, on=['id', 'subgroup'], how='left')
df_train = make_train_data(df_merged, 'id', 'subgroup')
df_train['subgroup'] = pd.Categorical(df_train['subgroup'], categories=range(2))

# Compile the model
print(" Compiling the model...")
priors = {
	'rate': PenalisedTensorSpline2D(
	type='global',
	symmetric=True,
	grid_type='diff-age',
	neighborhood=8
),
	'subgroup': PenalisedTensorSpline2D(
	loc=age_dist_props['subgroup'],
	type='partial',
	event_dim=2,
	transform='ilr',
	neighborhood=8
)
}

In [ ]:
df_full = df_train

In [ ]:
import xarray as xr
import jax.numpy as jnp

In [ ]:
ds = xr.Dataset(
			{
				'y': ('index', df_full['y'].astype(int).to_numpy()),
				'log_N': ('index', jnp.log(df_full['N'].to_numpy())),
				'log_P': ('age', jnp.log(data['age_dist'])),
				'aid': ('index', df_full['age_part'].to_numpy()),
                'bid':('index', df_full['age_cnt'].to_numpy())
			},
			coords={
				'index': df_full.index.to_numpy(),
				'age': ('age', np.arange(0, 81, dtype=int))
			}
		)

In [ ]:

model = HiBRCfine(ds, df_train, age_dist_props, priors)
model.print_model_shape()

# Fit the model
print(" Fitting the model...")
prng_key = jax.random.PRNGKey(0)
init_values = {'baseline': -model.log_P.mean()}

guide = AutoNormal(model.model, init_loc_fn=init_to_value(values=init_values))
model.run_inference_svi(prng_key, guide)

# Save the model
print(" Saving the model...")
model_path = 'HiBRC/model.pkl'

with open(model_path, 'wb') as f:
	pickle.dump(model, f)
print(f" Model saved to {model_path}\n")



In [ ]:

# ========== Summarise the results ==========
print(" Summarising the results...")
summariser = ModelSummariserSVI(model)

print(" Evaluating the model...")
evaluator = ModelEvaluator(summariser, data['cint_matrices'])
df_cint_eval = evaluator.evaluate_cint()
df_mcint_eval = evaluator.evaluate_mcint()

df_cint_eval.to_csv(f'HiBRC/cint_eval_{500}_{2}.csv', index=False)
df_mcint_eval.to_csv(f'HiBRC/mcint_eval_{500}_{2}.csv', index=False)

print(" Visualising the results...")
visualiser = ModelVisualiser(summariser)

# Plot the contact intensity matrices
chart_dict = visualiser.plot_cint()
fig_path = 'HiBRC/fig/post_cint_matrix_500_2.png'
chart_dict['subgroup'].save(fig_path)

# Plot marginal contact intensity
chart_dict = visualiser.plot_mcint(evaluator)
fig_path = 'HiBRC/fig/post_cint_marginal_500_2.png'
chart_dict['subgroup'].save(fig_path)